## Import 

In [2]:
from torch import nn,optim
import torch
from torch.utils.data import DataLoader,Dataset,TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

## Data

In [3]:
df = pd.read_csv('./diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [ ]:
no_zero = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

for feature in no_zero:
     df[feature] = df[feature].replace(0,np.nan)
     mean = df[feature].mean()
     df[feature] = df[feature].replace(np.nan,mean)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50,1
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31,0
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32,1
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21,0
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33,1


In [5]:
X = df.drop('Outcome',axis=1)
X.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148.0,72.0,35.00000,155.548223,33.6,0.627,50
1,1,85.0,66.0,29.00000,155.548223,26.6,0.351,31
2,8,183.0,64.0,29.15342,155.548223,23.3,0.672,32
3,1,89.0,66.0,23.00000,94.000000,28.1,0.167,21
4,0,137.0,40.0,35.00000,168.000000,43.1,2.288,33


In [19]:
y = np.array(df['Outcome'])
y[:5]

array([1, 0, 1, 0, 1])

In [20]:
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=12,stratify=y)
X_train , X_valid , y_train , y_valid = train_test_split(X_train,y_train,test_size=0.09,random_state=12,stratify=y_train)

X_train.shape , X_valid.shape , X_test.shape , y_train.shape ,y_valid.shape , y_test.shape

((558, 8), (56, 8), (154, 8), (558,), (56,), (154,))

In [21]:
X_scaler = StandardScaler()

X_train = X_scaler.fit_transform(X_train)
X_valid = X_scaler.transform(X_valid)
X_test = X_scaler.transform(X_test)

X_train.shape , X_valid.shape , X_test.shape , y_train.shape ,y_valid.shape , y_test.shape

((558, 8), (56, 8), (154, 8), (558,), (56,), (154,))

In [22]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_valid = torch.tensor(y_valid, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

X_train.shape , X_valid.shape , X_test.shape , y_train.shape ,y_valid.shape , y_test.shape

(torch.Size([558, 8]),
 torch.Size([56, 8]),
 torch.Size([154, 8]),
 torch.Size([558]),
 torch.Size([56]),
 torch.Size([154]))

In [23]:
train_set = TensorDataset(X_train,y_train)
valid_set = TensorDataset(X_valid,y_valid)
test_set = TensorDataset(X_test,y_test)

In [24]:
train_loader = DataLoader(train_set,batch_size=100,shuffle=True)
valid_loader = DataLoader(valid_set,batch_size=10,shuffle=True)
test_loader = DataLoader(test_set,batch_size=200,shuffle=True)

## Model and Training and Validation

In [62]:
class LogisticRegression(nn.Module):
    def __init__(self,in_features) -> None:
        super(LogisticRegression,self).__init__()
        self.linear = nn.Linear(in_features,1)
    def forward(self,X):
        return torch.sigmoid(self.linear(X))

model = LogisticRegression(X_train[0].shape[0])

In [63]:
optimizer = optim.SGD(model.parameters(),lr=0.2,momentum=0.9)
get_error = nn.BCELoss()

In [ ]:
train_hist_loss = []
train_hist_acc = []
valid_hist_loss = []
valid_hist_acc = []
n_epoches = 200
best_error = torch.inf

for i in range(n_epoches):

    mean_train_loss = 0
    mean_train_acc = 0
    mean_valid_loss = 0
    mean_valid_acc = 0

    for X_batch, y_batch in train_loader:
        y_hat = model(X_batch).squeeze()
        loss = get_error(y_hat , y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if loss < best_error:
            torch.save(model, './LogisticRegressionModel.pt')

        mean_train_loss += loss.item()
        mean_train_acc += torch.sum((y_hat > 0.5).float() == y_batch).item()

    mean_train_loss = mean_train_loss / len(train_loader)
    mean_train_acc = mean_train_acc / len(train_set)

    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            y_hat = model(X_batch).squeeze()
            loss = get_error(y_hat , y_batch)

            mean_valid_loss += loss.item()
            mean_valid_acc += torch.sum((y_hat > 0.5).float() == y_batch).item()

    mean_valid_loss = mean_valid_loss / len(valid_loader)
    mean_valid_acc = mean_valid_acc / len(valid_set)

    train_hist_loss.append(mean_train_loss)
    train_hist_acc.append(mean_train_acc)
    valid_hist_loss.append(mean_valid_loss)
    valid_hist_acc.append(mean_valid_acc)

    print(f"Epoch [{i+1:3d}/{n_epoches}]\t"
          f"Train Loss: {mean_train_loss}\t"
          f"Train Acc: {mean_train_acc}\t"
          f"Valid Loss: {mean_valid_loss}\t"
          f"Valid Acc: {mean_valid_acc:}\t"    )

Epoch [  1/200]	Train Loss: 0.4698077191909154	Train Acc: 0.7759856630824373	Valid Loss: 0.4586931864420573	Valid Acc: 0.7678571428571429	
Epoch [  2/200]	Train Loss: 0.4698358823855718	Train Acc: 0.7688172043010753	Valid Loss: 0.4705205609401067	Valid Acc: 0.7678571428571429	
Epoch [  3/200]	Train Loss: 0.4673782289028168	Train Acc: 0.7777777777777778	Valid Loss: 0.4498865182201068	Valid Acc: 0.7678571428571429	
Epoch [  4/200]	Train Loss: 0.47235750158627826	Train Acc: 0.7759856630824373	Valid Loss: 0.4663800075650215	Valid Acc: 0.7678571428571429	
Epoch [  5/200]	Train Loss: 0.4733882298072179	Train Acc: 0.7688172043010753	Valid Loss: 0.48021020988623303	Valid Acc: 0.7678571428571429	
Epoch [  6/200]	Train Loss: 0.46825064222017926	Train Acc: 0.7724014336917563	Valid Loss: 0.47061507403850555	Valid Acc: 0.75	
Epoch [  7/200]	Train Loss: 0.47680220007896423	Train Acc: 0.7741935483870968	Valid Loss: 0.4719797770182292	Valid Acc: 0.7321428571428571	
Epoch [  8/200]	Train Loss: 0.465804

## Test Model

In [67]:
model = torch.load('./LogisticRegressionModel.pt', weights_only=False)

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        y_hat = model(X_batch)
        loss = nn.functional.l1_loss(y_hat, y_batch)
        acc = torch.sum((y_hat > 0.5).float() == y_batch).item() / len(test_set)
print(loss)
print(acc)

tensor(0.4579)
85.96103896103897
